# Multistream Matrix-Fractal vs Binary Serial

This notebook reproduces the multistream comparison used by the article assets.

The comparison uses equal physical stream counts for both methods. A large payload is split across `N` streams. For the matrix-fractal method, each stream restarts its period hierarchy from the fastest digit channel.

## Interpretation

`Binary serial` is an ideal synchronized digital reference: one bit per stream tick with an external clock.

`Matrix-fractal 4x4` is the current canonical period-shift generator implemented by `MatrixFractalNumber`. It is not hand-estimated; latency is computed from generated period-shift cells as:

```text
required_payload_ticks = max(cell.shift_ticks + cell.period_ticks for cell in cells) + 1
```

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'snn_framework').exists():
    ROOT = ROOT.parents[1]
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt

from article_assets.scripts.generate_snn_comparative_metrics import comparative_rows

rows = comparative_rows()
df = pd.DataFrame(rows)
df.head(12)

## Large Payload Example

The table below shows how both methods scale for a large 2048-bit payload as the number of physical streams increases.

In [ ]:
df[df['payload_bits'] == 2048][[
    'label',
    'payload_bits',
    'physical_streams',
    'segment_bits',
    'digit_count_per_stream',
    'latency_ticks',
    'stream_ticks',
    'id_bits_per_stream_tick',
]]

## Plot Latency and Information Density

These plots mirror the generated article figures `figure_3_comparative_latency.png` and `figure_4_comparative_id.png`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for (method, streams), group in df.groupby(['method', 'physical_streams']):
    group = group.sort_values('payload_bits')
    label = group.iloc[0]['label']
    linestyle = '--' if method == 'binary_serial_multistream' else '-'
    marker = '^' if method == 'binary_serial_multistream' else 's'
    axes[0].plot(group['payload_bits'], group['latency_ticks'], label=label, linestyle=linestyle, marker=marker)
    axes[1].plot(group['payload_bits'], group['id_bits_per_stream_tick'], label=label, linestyle=linestyle, marker=marker)

axes[0].set_xscale('log', base=2)
axes[0].set_yscale('log', base=10)
axes[0].set_xlabel('Payload size, bits')
axes[0].set_ylabel('Latency, ticks')
axes[0].set_title('Multistream latency')
axes[0].grid(True, which='both', alpha=0.3)

axes[1].set_xscale('log', base=2)
axes[1].set_yscale('log', base=10)
axes[1].set_xlabel('Payload size, bits')
axes[1].set_ylabel('ID, bits / stream-tick')
axes[1].set_title('Multistream information density')
axes[1].grid(True, which='both', alpha=0.3)

axes[1].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
fig.tight_layout()
plt.show()

## Key Result

Multistreaming reduces latency for both binary and matrix-fractal transmission. In the current canonical step generator, binary serial remains the ideal synchronized lower-latency reference, while matrix-fractal transmission provides a self-structured period-shift payload representation that does not rely on bit position alone.

This notebook is therefore a resource-accounting experiment, not a claim that the current step generator is the final high-density physical channel.